# Plan2BoQ — Train YOLOv8 on Google Colab

This notebook trains a **door** and **window** detector for floor plan images.

## Before you start

1. **Runtime → Change runtime type → GPU** (T4 is enough; A100 is faster).
2. Prepare a YOLO dataset locally with `ml/dataset.py`, then zip the **folder** that contains `data.yaml`:
   ```
   floorplans_dataset/
     data.yaml
     images/train/   *.png
     images/val/     *.png
     labels/train/   *.txt  (YOLO: class cx cy w h)
     labels/val/     *.txt
   ```
3. Annotate labels with [LabelImg](https://github.com/HumanSignal/labelImg) (YOLO format), [CVAT](https://cvat.org/), or [Roboflow](https://roboflow.com/) — classes **0 = door**, **1 = window**.

## After training

Download `best.pt` from the last cell and place it in your repo as `ml/best.pt`. Then `process_cad.py` / `process_floor_plans.py` will run ML detection automatically.

## 1. Environment & GPU

In [ ]:
# @title Install dependencies { display-mode: "form" }
!pip install -q ultralytics pyyaml

import sys
import torch

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

!nvidia-smi -L 2>/dev/null || echo "(nvidia-smi not found — use GPU runtime)"

import ultralytics
ultralytics.checks()

## 2. Mount Google Drive (optional)

Use this if your dataset zip is already on Drive (e.g. `MyDrive/Plan2BoQ/dataset.zip`).

In [ ]:
# @title Mount Drive { display-mode: "form" }
MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ModuleNotFoundError:
        print("Not running in Colab — skipping Drive mount.")
else:
    print("Drive mount skipped.")

## 3. Load dataset

Choose **one** source below.

In [ ]:
# @title Dataset source { display-mode: "form" }
from pathlib import Path
from typing import Optional
import shutil
import zipfile

# upload_zip | drive_zip | drive_folder
DATASET_MODE = "upload_zip"  # @param ["upload_zip", "drive_zip", "drive_folder"]

# If drive_zip: path to .zip on Drive (after mounting)
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Plan2BoQ/floorplans_dataset.zip"  # @param {type:"string"}

# If drive_folder: folder that already contains data.yaml
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/Plan2BoQ/floorplans_dataset"  # @param {type:"string"}

WORK_ROOT = Path("/content/plan2boq_yolo")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def find_data_yaml(root: Path) -> Optional[Path]:
    direct = root / "data.yaml"
    if direct.exists():
        return direct
    for p in root.rglob("data.yaml"):
        return p
    return None

if DATASET_MODE == "upload_zip":
    try:
        from google.colab import files
        print("Upload your dataset .zip (contains data.yaml + images + labels)...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        extract_to = WORK_ROOT / "extracted"
        extract_to.mkdir(parents=True, exist_ok=True)
        for name in uploaded:
            if not str(name).lower().endswith(".zip"):
                print("Skipping non-zip:", name)
                continue
            zpath = WORK_ROOT / name
            with open(zpath, "wb") as f:
                f.write(uploaded[name])
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall(extract_to)
            print("Extracted:", name, "->", extract_to)
        DATA_YAML = find_data_yaml(extract_to)
    except ModuleNotFoundError:
        raise RuntimeError(
            "upload_zip only works in Google Colab. Use drive_zip or drive_folder, "
            "or place files under /content/plan2boq_yolo/extracted manually."
        )

elif DATASET_MODE == "drive_zip":
    src = Path(DRIVE_ZIP_PATH)
    if not src.exists():
        raise FileNotFoundError(f"Zip not found: {src}")
    extract_to = WORK_ROOT / "extracted"
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src, "r") as zf:
        zf.extractall(extract_to)
    DATA_YAML = find_data_yaml(extract_to)

else:  # drive_folder
    folder = Path(DRIVE_FOLDER_PATH)
    if not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
    DATA_YAML = find_data_yaml(folder)

if DATA_YAML is None:
    raise FileNotFoundError(
        "Could not find data.yaml. Ensure your zip extracts to a folder containing data.yaml "
        "(or a single subfolder that contains it)."
    )

DATASET_ROOT = DATA_YAML.parent
print("data.yaml:", DATA_YAML)
print("Dataset root:", DATASET_ROOT)

train_imgs = list((DATASET_ROOT / "images" / "train").glob("*.png"))
train_imgs += list((DATASET_ROOT / "images" / "train").glob("*.jpg"))
val_imgs = list((DATASET_ROOT / "images" / "val").glob("*.png"))
val_imgs += list((DATASET_ROOT / "images" / "val").glob("*.jpg"))
print(f"Training images: {len(train_imgs)}")
print(f"Validation images: {len(val_imgs)}")
if len(train_imgs) == 0:
    raise RuntimeError("No training images found under images/train/")

## 4. Train YOLOv8

Reduce **batch** if you hit CUDA OOM (try 4 or 2 on free T4 with `imgsz=1280`).

In [ ]:
# @title Training hyperparameters { display-mode: "form" }
YOLO_BASE = "yolov8s.pt"  # @param ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]
EPOCHS = 100  # @param {type:"integer"}
IMG_SIZE = 1280  # @param {type:"integer"}
BATCH = 8  # @param {type:"integer"}
PATIENCE = 20  # @param {type:"integer"}
RUN_NAME = "door_window_v1"  # @param {type:"string"}

from ultralytics import YOLO

model = YOLO(YOLO_BASE)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    project="runs/floorplan",
    name=RUN_NAME,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    degrees=5.0,
    scale=0.3,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
)

run_dir = Path("runs/floorplan") / RUN_NAME
print("Run directory:", run_dir)

## 5. Validation metrics

In [ ]:
metrics = model.val()
box = metrics.box
print(f"mAP50:     {box.map50:.4f}")
print(f"mAP50-95:  {box.map:.4f}")
print(f"Precision: {box.mp:.4f}")
print(f"Recall:    {box.mr:.4f}")

names = list(model.names.values())
for i, name in enumerate(names):
    ap50 = float(box.ap50[i]) if hasattr(box, "ap50") and len(box.ap50) > i else 0.0
    ap = float(box.ap[i]) if hasattr(box, "ap") and len(box.ap) > i else 0.0
    print(f"  {name}: AP50={ap50:.4f}  AP50-95={ap:.4f}")

## 6. Training plots

In [ ]:
from IPython.display import Image, display

for plot in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
    p = run_dir / plot
    if p.exists():
        print("---", plot, "---")
        display(Image(filename=str(p), width=900))

## 7. Export & download `best.pt`

Save to Drive and/or download to your laptop, then copy into **`Plan2BoQ - PDF cleaning/ml/best.pt`**.

In [ ]:
import shutil

best_model = run_dir / "weights" / "best.pt"
if not best_model.exists():
    raise FileNotFoundError(f"Missing {best_model}")

export_path = WORK_ROOT / "best.pt"
shutil.copy2(str(best_model), str(export_path))
print("Exported:", export_path)
print("Size (MB):", export_path.stat().st_size / 1024 / 1024)

# Copy to Google Drive (optional)
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_OUT = "/content/drive/MyDrive/Plan2BoQ/best.pt"  # @param {type:"string"}

if SAVE_TO_DRIVE:
    outp = Path(DRIVE_OUT)
    outp.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(export_path), str(outp))
    print("Saved to Drive:", outp)

try:
    from google.colab import files
    files.download(str(export_path))
    print("Browser download started for best.pt")
except ModuleNotFoundError:
    print("Not in Colab — download step skipped.")

## 8. Quick test on validation images

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

infer_model = YOLO(str(export_path))
val_images = sorted((DATASET_ROOT / "images" / "val").glob("*.png"))
val_images += sorted((DATASET_ROOT / "images" / "val").glob("*.jpg"))
val_images = val_images[:5]

if not val_images:
    print("No val images — skipping inference demo.")
else:
    pred_dir = WORK_ROOT / "test_predictions"
    pred_dir.mkdir(exist_ok=True)
    for img_path in val_images:
        results = infer_model.predict(
            source=str(img_path),
            conf=0.25,
            save=True,
            project=str(pred_dir.parent),
            name=pred_dir.name,
        )
        for r in results:
            boxes = r.boxes
            if boxes is None or len(boxes) == 0:
                print(img_path.name, ": 0 detections")
                continue
            door_c = sum(1 for b in boxes if infer_model.names[int(b.cls)] == "door")
            win_c = sum(1 for b in boxes if infer_model.names[int(b.cls)] == "window")
            print(f"{img_path.name}: {door_c} doors, {win_c} windows")

    out_imgs = list(pred_dir.glob("*.jpg")) + list(pred_dir.glob("*.png"))
    for img in out_imgs[:5]:
        display(Image(filename=str(img), width=900))